# Pacotes

In [1]:
import pandas as pd

from sklearn import  model_selection
from sklearn.model_selection import RandomizedSearchCV 
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import confusion_matrix

from sklearn import tree
from matplotlib import pyplot as plt

from sklearn.model_selection import KFold 
from sklearn.model_selection import cross_val_score 

import numpy as np
import seaborn as sns
from prettytable import PrettyTable
import sklearn

# Funções

In [2]:
##############################
###### Matriz de confusão ####
##############################

def matriz_confusao(y_real,y_predito,modelo):

### Grafico ###

  tabela=confusion_matrix(y_real,y_predito)

  group_names = ["True Neg","False Pos","False Neg","True Pos"]
  group_counts = ["{0:0.0f}".format(value) for value in
                tabela.flatten()]
  group_percentages = ["{0:.5%}".format(value) for value in
                     tabela.flatten()/np.sum(tabela)]
  labels = [f"{v1}\n{v2}\n{v3}" for v1, v2, v3 in
          zip(group_names,group_counts,group_percentages)]
  labels = np.asarray(labels).reshape(2,2)
  f = plt.figure()
  f.set_figwidth(8)
  f.set_figheight(8)

  sns.heatmap(tabela, annot=labels, fmt="", cmap='Blues')

### Tabela ###
  Resultados=PrettyTable()
  Resultados.field_names=["Métrica","Resultado"]
  Resultados.title= modelo
  Resultados.align["Métrica"]="l"
  Resultados.align["Resultado"]="r"

  Resultados.add_row(["Acurácia:",round(sklearn.metrics.accuracy_score(y_real,y_predito),2)])
  Resultados.add_row(["Precisão:",round(sklearn.metrics.precision_score(y_real,y_predito),2)])
  Resultados.add_row(["Recall:",round(sklearn.metrics.recall_score(y_real,y_predito),2)])
  Resultados.add_row(["F1-Score:",round(sklearn.metrics.f1_score(y_real,y_predito),2)])

  print(Resultados)
  
  return

# Modelo

In [21]:
for i in range(18):

    print('')
    print('#_______________ Arquivo ', i, '________________#')
    print('')

    i=str(i)


    # Extração dos dados


    X_treino = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_ins_over.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

    #X_treino = X_treino[(X_treino.loc[:,'flg_comunidades' ] == 0)]

    print('')
    print('Tamanho do treino e quantidade de ocorrências')
    print(len(X_treino.index))
    print(round(sum(X_treino['flg_ocorrencia'])))
    print('')

    X_teste = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_oos.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

    #X_teste = X_teste[(X_teste.loc[:,'flg_comunidades' ] == 0)]

    print('')
    print('Tamanho do teste e quantidade de ocorrências')
    print(len(X_teste.index))
    print(round(sum(X_teste['flg_ocorrencia'])))
    print('')

    oot = pd.read_csv('C:/Users/debor/Documents/PDPA/Dados/CSV/df'+i+'_final_oot.csv'
                     ,sep = ','
                     ,decimal='.'
                      )

    #oot = oot[(oot.loc[:,'flg_comunidades' ] == 0)]

    print('')
    print('Tamanho do teste 2 meses a frente e quantidade de ocorrências')
    print(len(oot.index))
    print(round(sum(oot['flg_ocorrencia'])))
    print('')

    if round(sum(X_treino['flg_ocorrencia'])) == 0 :
        continue

    if round(sum(X_teste['flg_ocorrencia'])) == 0 :
        continue

    if round(sum(oot['flg_ocorrencia'])) == 0 :
        continue


    #Ajuste na flg_ocorrencia

    y_treino = X_treino['flg_ocorrencia']
    X_treino = X_treino.drop('flg_ocorrencia', axis=1)
    n = len(X_treino.columns)

    y_teste = X_teste['flg_ocorrencia']
    X_teste = X_teste.drop('flg_ocorrencia', axis=1)
    
    #############################################
    ############ Floresta de Decisão ############
    #############################################

    maxi = int(sum(y_treino)-1) if sum(y_treino) < 10 else 10
    min_s = int((sum(y_treino)/10)+1) if sum(y_treino) < 300 else 30

    classificador_floresta = RandomForestClassifier()

    lista_parametros = {
        'n_estimators': [i + 1 for i in range(10,200,1)],
        'criterion': ['gini', 'entropy', 'log_loss'],
        'max_depth': [i + 1 for i in range(2,30,1)],
        'min_samples_split': [i + 1 for i in range(1,int(maxi-1),1)],
        'max_features': ['sqrt', 'log2', None],
        'max_leaf_nodes': [i + 1 for i in range(10,len(y_treino),1)],
        'min_impurity_decrease': stats.uniform(0, 1),
        'bootstrap' :[True]
        }

    rand_search = RandomizedSearchCV(classificador_floresta, 
                                     param_distributions = lista_parametros,
                                     cv = int(maxi-1),
                                     random_state = 42,
                                     scoring = 'f1') 
    rand_search.fit(X_treino.iloc[:,3:n], y_treino) 


    print('')
    print(rand_search.best_estimator_)
    print('')

    classificador_floresta = rand_search.best_estimator_
    classificador_floresta.fit(X_treino.iloc[:,3:n], y_treino) 

    #############################################
    ############ Qualidade do modelo ############
    #############################################


    kfold  = KFold(n_splits=maxi, shuffle=False)
    result_floresta = cross_val_score(classificador_floresta, X_treino.iloc[:,3:n], y_treino, cv = kfold)
    #K-fold
    #'Média da acurácia:'
    print(np.mean(result_floresta))
    #'Variância da acurácia'
    print(np.std(result_floresta))

    #Teste - OOS
    y_teste, classificador_floresta.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(y_teste, classificador_floresta.predict(X_teste.iloc[:,3:n])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(y_teste, classificador_floresta.predict(X_teste.iloc[:,3:n])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(y_teste, classificador_floresta.predict(X_teste.iloc[:,3:n])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(y_teste, classificador_floresta.predict(X_teste.iloc[:,3:n])),2))

    #Teste - OOT
    y_teste, classificador_floresta.predict(X_teste.iloc[:,3:n])
    #"Acurácia:"
    print(round(sklearn.metrics.accuracy_score(oot.iloc[:,0], classificador_floresta.predict(oot.iloc[:,4:n+1])),2))
    #"Precisão:"
    print(round(sklearn.metrics.precision_score(oot.iloc[:,0], classificador_floresta.predict(oot.iloc[:,4:n+1])),2))
    #"Recall:"
    print(round(sklearn.metrics.recall_score(oot.iloc[:,0], classificador_floresta.predict(oot.iloc[:,4:n+1])),2))
    #"F1-Score:"
    print(round(sklearn.metrics.f1_score(oot.iloc[:,0], classificador_floresta.predict(oot.iloc[:,4:n+1])),2))


#_______________ Arquivo  0 ________________#


Tamanho do treino e quantidade de ocorrências
6934
3467


Tamanho do teste e quantidade de ocorrências
1771
3


Tamanho do teste 2 meses a frente e quantidade de ocorrências
24890
35


RandomForestClassifier(criterion='log_loss', max_depth=28, max_features=None,
                       max_leaf_nodes=5345,
                       min_impurity_decrease=0.05808361216819946,
                       min_samples_split=9, n_estimators=127)

0.9504164743357826
0.04863717875847233
0.9
0.02
1.0
0.03
0.88
0.0
0.11
0.0

#_______________ Arquivo  1 ________________#


Tamanho do treino e quantidade de ocorrências
31074
15537


Tamanho do teste e quantidade de ocorrências
7740
15


Tamanho do teste 2 meses a frente e quantidade de ocorrências
135360
230


RandomForestClassifier(criterion='log_loss', max_depth=28, max_features=None,
                       max_leaf_nodes=29921,
                       min_impurity_decrease=0.05808361216819946,
            


KeyboardInterrupt



In [7]:
np.unique(y_treino)

array([0., 1.])

In [11]:
np.unique(oot.iloc[:,n])

array([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12], dtype=int64)

In [19]:
oot.columns[n]

'mes_ocorrencia'

In [14]:
n

18

In [20]:
oot

,flg_ocorrencia,id_tempo_x,id_tempo_y,id_solicitacao,noventaseishoras,mes,Altitude_numerica,Declividade_numerica,Curv_Vertical_numerica,Declividade_classes_C,Orientacao_octantes,Forma_de_terreno_classes,Relevo_sombreado_numerico,ADD_divisores_talvegues,flg_comunidades,flg_cobertura_vegetal,flg_areas_de_risco,Acoes_DC_0,mes_ocorrencia
0,1.0,11621311,11621311.0,360219.0,81.6,100.0,72.835698,24.700539,0.101781,4.0,2.671637,9.000000,0.673064,17.674506,1,1,0,0,2
1,1.0,11621311,11621324.0,170219.0,71.4,121.2,93.659164,13.049868,-0.021418,3.0,4.779852,4.487377,0.416087,7.779852,0,0,0,0,2
2,1.0,11621326,11621331.0,180219.0,81.6,100.0,72.835698,24.700539,0.101781,4.0,2.671637,9.000000,0.673064,17.674506,1,1,0,0,2
3,1.0,11621386,11621393.0,220219.0,67.2,110.0,47.927171,25.896740,-0.081723,4.0,4.034755,4.000000,0.185556,7.034755,1,0,0,0,2
4,1.0,11621521,11621521.0,400219.0,78.4,128.2,84.692587,21.248526,-0.080315,4.0,5.086578,1.000000,0.226658,8.787685,1,1,0,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24885,0.0,13197061,11656893.0,130319.0,59.4,362.2,67.212609,20.490922,0.061418,4.0,2.478355,9.000000,0.746501,18.000000,1,0,0,0,2
24886,0.0,13226926,11656893.0,130319.0,1.2,188.0,67.212609,20.490922,0.061418,4.0,2.478355,9.000000,0.746501,18.000000,1,0,0,0,2
24887,0.0,13227151,11656893.0,130319.0,1.2,188.0,67.212609,20.490922,0.061418,4.0,2.478355,9.000000,0.746501,18.000000,1,0,0,0,2
24888,0.0,13227166,11656893.0,130319.0,1.2,188.0,67.212609,20.490922,0.061418,4.0,2.478355,9.000000,0.746501,18.000000,1,0,0,0,2
